# ChurnGuard — K-Nearest Neighbors (KNN) Model (COMPE 510)

This notebook trains and evaluates a **K-Nearest Neighbors (KNN)** classifier to predict **customer churn** using the **Telco Customer Churn** dataset.

**Input file expected:** `WA_Fn-UseC_-Telco-Customer-Churn.csv`

**Outputs (saved for deployment):**
- `churn_model_knn.joblib` (trained model + preprocessing pipeline)
- `model_columns_knn.joblib` (feature names after preprocessing)


## 1) Setup & Imports

Run this cell first. It installs any missing dependencies (safe to re-run).


In [ ]:
!pip -q install -U scikit-learn joblib

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

## 2) Load & Inspect Data

We load the dataset, coerce `TotalCharges` to numeric (some rows contain blanks), and drop rows with missing values created by the conversion.


In [ ]:
# Load dataset (expects the CSV to be in the same working directory)
csv_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(csv_path)

# Clean TotalCharges (often has blanks)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop rows made NaN by coercion (and any other NaNs)
df = df.dropna().reset_index(drop=True)

df.head()


In [ ]:
df.info()


## 3) Preprocessing

KNN is distance-based, so **feature scaling matters**.  
We use a scikit-learn `Pipeline` + `ColumnTransformer` to:
- **One-hot encode** categorical columns
- **Scale** numeric columns

This keeps preprocessing consistent for training and future predictions.


In [ ]:
# Separate target and features
target_col = "Churn"
if target_col not in df.columns:
    raise ValueError(f"Expected target column '{target_col}' not found in CSV. Columns: {list(df.columns)}")

X = df.drop(columns=[target_col])
y = df[target_col].map({"Yes": 1, "No": 0})  # binary encode

# Drop customerID if present (identifier, not predictive)
if "customerID" in X.columns:
    X = X.drop(columns=["customerID"])

# Identify numeric vs categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_features, categorical_features[:10], len(categorical_features)


In [ ]:
# Preprocessor: scale numeric, one-hot encode categorical
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline(steps=[("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ],
    remainder="drop"
)


## 4) Train/Test Split & Model Training

We split into train/test sets and tune a few sensible K values (neighbors).


In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


In [ ]:

k_values = [3, 5, 7, 9, 11, 15, 21]
results = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("knn", knn)
    ])
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results.append((k, acc))

results_df = pd.DataFrame(results, columns=["k", "accuracy"]).sort_values("accuracy", ascending=False)
results_df


In [ ]:
# Select best K and retrain final model
best_k = int(results_df.iloc[0]["k"])
print("Best k:", best_k)

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier(n_neighbors=best_k))
])

final_model.fit(X_train, y_train)


## 5) Evaluation

We report:
- Accuracy
- Classification report (precision/recall/F1)
- Confusion matrix


In [ ]:
y_pred = final_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


## 6) Save Model Artifacts

We save the full preprocessing + model pipeline with `joblib`, plus the final feature names after preprocessing.
If running on **Google Colab**, the last cell will trigger downloads.


In [ ]:
# Save the full pipeline (preprocessing + KNN)
joblib.dump(final_model, "churn_model_knn.joblib")

# Extract feature names after preprocessing for reference (useful for debugging/inference)
ohe = final_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["ohe"]
cat_feature_names = ohe.get_feature_names_out(categorical_features)
model_columns = np.concatenate([numeric_features, cat_feature_names]).tolist()

joblib.dump(model_columns, "model_columns_knn.joblib")

print("Saved:")
print("- churn_model_knn.joblib")
print("- model_columns_knn.joblib")
print("Number of final features:", len(model_columns))


In [ ]:
try:
    from google.colab import files
    files.download("churn_model_knn.joblib")
    files.download("model_columns_knn.joblib")
except Exception as e:
    print("Not running in Colab (or downloads not available). That's okay.")
    print("Error (informational):", e)
